In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="thucdangvan020999/singlish-speaker2050", 
                  repo_type="dataset", local_dir="./singlish-speaker2050")
snapshot_download(repo_id="thucdangvan020999/singlish-speaker2202", 
                  repo_type="dataset", local_dir="./singlish-speaker2202")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:01<00:00,  1.85it/s]


'/home/ubuntu/singlish-speaker2202'

In [3]:
files = glob('singlish-speaker*/*/*.parquet')
files

['singlish-speaker2202/data/train-00000-of-00001.parquet',
 'singlish-speaker2050/data/train-00000-of-00001.parquet']

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 880/880 [00:26<00:00, 33.04it/s]


In [7]:
len(data)

1749

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'singlish-speaker2202_audio/singlish-speaker2202-data-train-00000-of-00001_0.mp3',
 'text': '11, 12, 13, 21, 32, 43, 50,',
 'speaker': 'singlish-speaker2202_audio'}

In [9]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'singlish-speaker')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 878.02ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 76.1kB / 76.1kB,  245kB/s  
Processing Files (1 / 1): 100%|██████████| 76.1kB / 76.1kB,  190kB/s  
New Data Upload: 100%|██████████| 76.1kB / 76.1kB,  190kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.10 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c4a7effc5c14872f55fadc8b13c5954f80c116ab', commit_message='Upload dataset', commit_description='', oid='c4a7effc5c14872f55fadc8b13c5954f80c116ab', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('singlish-speaker-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
folders = glob('singlish-speaker*_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

singlish-speaker2202_audio_neucodec
singlish-speaker2050_audio_neucodec
singlish-speaker2202_audio
singlish-speaker2050_audio


In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('singlish-speaker*_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▊| 22.6MB / 23.0MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 23.0MB / 23.0MB,  316kB/s  
Processing Files (1 / 1): 100%|██████████| 23.0MB / 23.0MB,  263kB/s  
New Data Upload: 100%|██████████| 23.0MB / 23.0MB,  263kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  920kB /  920kB,   ???B/s  
Processing Files (1 / 1): 100%|██████████|  920kB /  920kB,  0.00B/s  
New Data Upload: 100%|██████████|  920kB /  920kB,  0.00B/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▉| 20.0MB / 20.1MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 20.1MB / 20.1MB,  119kB/s  
Processing Files (1 / 1): 100%|██████████| 20.1MB / 20.1MB, 99.1kB/s  
New Data Upload: 100%|██████████| 20.1MB / 20.1MB, 99.1kB/s  
